In [ ]:
import csv, os
from collections import defaultdict, Counter
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

DATA_DIR = "./"
OUT_PATH = "incidents_classified.csv"

f_perils = os.path.join(DATA_DIR, "AIRiskPerils.csv")
f_lob    = os.path.join(DATA_DIR, "LineOfBusiness.csv")
f_mit    = os.path.join(DATA_DIR, "../mongodump_full_snapshot/classifications_MIT.csv")
f_inc    = os.path.join(DATA_DIR, "../mongodump_full_snapshot/incidents.csv")

In [ ]:
peril_desc = {}
peril_terms = defaultdict(list)
current_top = None

with open(f_perils, encoding="utf-8-sig") as f:
    for row in csv.DictReader(f):
        lvl  = row.get("Level", "").strip()
        risk = row.get("Risk", "").strip()
        sub  = row.get("Sub-Risk", "").strip()
        cat  = row.get("Sub-Category", "").strip()
        desc = row.get("Description", "").strip()
        if lvl == "Top" and risk:
            current_top = risk
            peril_desc[risk] = desc
        for phrase in (sub, cat):
            if phrase and current_top:
                peril_terms[current_top].append(phrase)

PERILS = list(peril_desc.keys())
print("Top-level perils:")
for p in PERILS:
    print(" -", p)

Top-level perils:
 - Autonomous Actions
 - Bias & Fairness
 - Privacy, Confidentiality & Infringement
 - Reliability
 - Security & Misuse
 - Governance, Oversight & Explainability


In [ ]:
# Keyword signatures per peril
KW = {
 "Autonomous Actions": ["autonomous","self-driving","driverless","robot","robotic","drone","uav","unmanned","tool use","tool-use","multi-agent","agentic","autonomous agent","autonomous vehicle","autopilot","self-navigat","trading bot","algorithmic trading","autonomous weapon","loss of control","runaway","unintended action","reward hacking"],
 "Bias & Fairness": ["bias","biased","discriminat","racist","racial bias","sexist","gender bias","ethnic","fairness","unfair","disparate impact","disparate treatment","protected class","protected attribute","stereotyp","inequit","unequal treatment","offensive","hate speech","recidivism","hiring bias","algorithmic bias","religious discriminat","ageism","representational harm","allocative harm","demographic parity"],
 "Privacy, Confidentiality & Infringement": ["privacy","personal data","confidential","surveillance","facial recognition","data leak","data breach","re-identif","deanonymi","copyright","intellectual property","infringement","personally identifiable","tracking","biometric","web scraping","data scraping","memorization","memorisation","unauthorized access","gdpr","right to be forgotten","consent violation","data exfiltration"],
 "Reliability": ["hallucinat","fabricat","inaccurate","malfunction","misdiagnos","unreliable","model drift","distribution shift","false result","incorrect output","factual error","factually wrong","overconfiden","miscalibrat","spurious correlation","edge case failure","silent failure","degradation","inconsistent output"],
 "Security & Misuse": ["hacking","cyberattack","malware","phishing","deepfake","fraud","scam","jailbreak","prompt injection","data poisoning","model poisoning","exploit","impersonat","adversarial example","adversarial attack","evasion attack","model extraction","model inversion","membership inference","backdoor","attack","disinformation","misinformation","social engineering","theft","weapon"],
 "Governance, Oversight & Explainability": ["transparen","interpretab","explainab", "accountab","human oversight","human-in-the-loop","disclosure","black box","auditab","audit trail","governance","undisclosed","opaque","traceab","provenance","documentation gap","lack of recourse","contestab","due process","regulatory compliance","non-compliance"],
}

# MIT Risk Domain priors
MIT_PRIOR = {
 "1. Discrimination and Toxicity": {"Bias & Fairness": 1.0},
 "2. Privacy & Security": {"Privacy, Confidentiality & Infringement": 0.7, "Security & Misuse": 0.5},
 "3. Misinformation": {"Reliability": 0.8, "Governance, Oversight & Explainability": 0.3},
 "4. Malicious Actors & Misuse": {"Security & Misuse": 1.0},
 "5. Human-Computer Interaction": {"Governance, Oversight & Explainability": 0.5, "Reliability": 0.4},
 "6. Socioeconomic & Environmental Harms": {"Governance, Oversight & Explainability": 0.5},
 "7. AI system safety, failures, and limitations": {"Reliability": 0.9, "Autonomous Actions": 0.4},
}

peril_docs = []
for p in PERILS:
    txt = " ".join([peril_desc[p]] + peril_terms[p]*2 + KW[p]*3)
    peril_docs.append(txt)

In [ ]:
# LOB IGNORE
lob_agg = defaultdict(lambda: defaultdict(lambda: [0, 0.0]))  # risk -> lob -> [yes_count, impact_sum]

with open(f_lob, encoding="utf-8-sig") as f:
    next(f)  # skip the title row above the real header
    for row in csv.DictReader(f):
        risk = row["Risk"].strip()
        lob  = row["Line of Business"].strip()
        if not risk or not lob:
            continue
        if row.get("Coverage Trigger (Y/N)", "").strip().lower() == "yes":
            lob_agg[risk][lob][0] += 1
        try:
            imp = float(row.get("Composite Impact", "") or 0)
        except ValueError:
            imp = 0.0
        lob_agg[risk][lob][1] += imp

best_lob   = {}
lob_ranked = {}
for risk, d in lob_agg.items():
    ranked = sorted(d.items(), key=lambda x: (-x[1][0], -x[1][1]))
    best_lob[risk]   = ranked[0][0]
    lob_ranked[risk] = [l for l, _ in ranked[:3]]

for p in PERILS:
    print(f"{p:45} -> {best_lob.get(p, '')}")

Autonomous Actions                            -> D&O
Bias & Fairness                               -> EPLI
Privacy, Confidentiality & Infringement       -> Cyber
Reliability                                   -> Tech E&O
Security & Misuse                             -> Crime / Fidelity
Governance, Oversight & Explainability        -> D&O


In [ ]:
mit = {}
with open(f_mit, encoding="utf-8-sig") as f:
    for row in csv.DictReader(f):
        mit[row["Incident ID"].strip()] = row

incidents = []
with open(f_inc, encoding="utf-8-sig") as f:
    for row in csv.DictReader(f):
        incidents.append(row)

def inc_text(r):
    return f"{r.get('title','')} {r.get('description','')}"

print(f"{len(incidents)} incidents, {len(mit)} MIT rows")
matched = sum(1 for r in incidents if r['incident_id'].strip() in mit)
print(f"{matched} incidents have a MIT classification")

1656 incidents, 1498 MIT rows
1498 incidents have a MIT classification


In [ ]:
corpus = peril_docs + [inc_text(r) for r in incidents]
vec = TfidfVectorizer(stop_words="english", ngram_range=(1, 2), min_df=1, max_features=20000)
X = vec.fit_transform(corpus)
Pm = X[:len(PERILS)]
Im = X[len(PERILS):]
sims = cosine_similarity(Im, Pm)

def kw_score(text):
    t = text.lower()
    return [sum(t.count(k) for k in KW[p]) for p in PERILS]

W_SIM, W_KW, W_PRIOR = 0.5, 0.3, 0.35

out = []
for i, r in enumerate(incidents):
    text = inc_text(r)

    sim = sims[i]
    sim_n = sim / (sim.max() if sim.max() > 0 else 1.0)

    kw = kw_score(text)
    kmax = max(kw) if max(kw) > 0 else 1
    kw_n = [k / kmax for k in kw]

    rec = mit.get(r["incident_id"].strip())
    dom = rec["Risk Domain"].strip() if rec else ""
    prior = [0.0] * len(PERILS)
    for p_name, w in MIT_PRIOR.get(dom, {}).items():
        prior[PERILS.index(p_name)] = w

    score = [W_SIM*sim_n[j] + W_KW*kw_n[j] + W_PRIOR*prior[j] for j in range(len(PERILS))]
    best = max(range(len(PERILS)), key=lambda j: score[j])
    peril = PERILS[best]

    ss = sorted(score, reverse=True)
    conf = round((ss[0] - ss[1]) / (ss[0] if ss[0] else 1), 3)

    out.append({
        "incident_id": r["incident_id"],
        "title": r.get("title", ""),
        "date": r.get("date", ""),
        "MIT_Risk_Domain": dom,
        "MIT_Risk_Subdomain": rec["Risk Subdomain"].strip() if rec else "",
        "AI_Risk_Peril": peril,
        "Line_of_Business": best_lob.get(peril, ""),
        "Alt_Lines_of_Business": "; ".join(lob_ranked.get(peril, [])),
        "Assignment_Confidence": conf,
        "description": r.get("description", ""),
    })

print("Classified", len(out), "incidents")

Classified 1656 incidents


In [21]:
cols = ["incident_id","title","date","MIT_Risk_Domain","MIT_Risk_Subdomain",
        "AI_Risk_Peril","Line_of_Business","Alt_Lines_of_Business",
        "Assignment_Confidence","description"]

with open(OUT_PATH, "w", newline="", encoding="utf-8-sig") as f:
    w = csv.DictWriter(f, fieldnames=cols)
    w.writeheader()
    w.writerows(out)

print("Wrote", OUT_PATH)
print("\nAI Risk Peril distribution:")
for k, v in Counter(o["AI_Risk_Peril"] for o in out).most_common():
    print(f"  {v:4}  {k}")
print("\nLine of Business distribution:")
for k, v in Counter(o["Line_of_Business"] for o in out).most_common():
    print(f"  {v:4}  {k}")

Wrote incidents_classified.csv

AI Risk Peril distribution:
   673  Security & Misuse
   239  Privacy, Confidentiality & Infringement
   238  Autonomous Actions
   229  Reliability
   185  Bias & Fairness
    92  Governance, Oversight & Explainability

Line of Business distribution:
   673  Crime / Fidelity
   330  D&O
   239  Cyber
   229  Tech E&O
   185  EPLI
